# DSO 530: Regression Models — LC & HALC Prediction

This notebook builds and compares Tweedie regression models to predict:
- **LC** (Loss Cost per Exposure Unit) = X.15 / X.16
- **HALC** (Historically Adjusted Loss Cost) = (X.15 / X.16) × X.18

Models compared: **Tweedie GLM**, **XGBoost**, **LightGBM**, and an **Ensemble**.

Evaluation metric: **out-of-sample MSE**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb
from lightgbm import LGBMRegressor

np.random.seed(42)
print('Libraries loaded.')

In [ ]:
BASE = 'data/processed/'

X_full = pd.read_csv(BASE + 'train_X.csv')
y_full = pd.read_csv(BASE + 'train_y.csv')
X_test = pd.read_csv(BASE + 'test_X.csv')

LC   = y_full['LC']
HALC = y_full['HALC']

print(f'Train: {X_full.shape} | Test: {X_test.shape}')
print(y_full[['LC', 'HALC']].describe().round(4))

## 1. Exploratory Analysis of Targets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, color in zip(axes, ['LC', 'HALC'], ['steelblue', 'coral']):
    vals = y_full[col]
    nonzero = vals[vals > 0]
    ax.hist(nonzero, bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{col} — non-zero distribution (n={len(nonzero):,})')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'LC   zero rate: {(LC == 0).mean():.2%}')
print(f'HALC zero rate: {(HALC == 0).mean():.2%}')
print(f'LC   max: {LC.max():.2f}  |  HALC max: {HALC.max():.2f}')

## 2. Train / Validation Split

In [ ]:
X_tr, X_val, lc_tr, lc_val, halc_tr, halc_val = train_test_split(
    X_full, LC, HALC, test_size=0.2, random_state=42
)

print(f'Train: {X_tr.shape} | Val: {X_val.shape}')
print(f'LC   train zero rate: {(lc_tr == 0).mean():.2%}')
print(f'HALC train zero rate: {(halc_tr == 0).mean():.2%}')

results = {}  # stores {model_name: val_MSE}

## 3. Model Training: LC

LC follows a Tweedie distribution (compound Poisson-Gamma, p ∈ (1,2)): large mass at zero plus a right-skewed positive tail.

### 3a. Tweedie GLM — Baseline

In [ ]:
# Grid search over power (p) and regularization (alpha)
best_glm_lc_mse = np.inf
best_glm_lc = None
best_glm_lc_params = {}

for p in [1.1, 1.3, 1.5, 1.7, 1.9]:
    for alpha in [0.001, 0.01, 0.1, 1.0]:
        glm = TweedieRegressor(power=p, alpha=alpha, max_iter=2000, link='log')
        glm.fit(X_tr, lc_tr)
        pred = glm.predict(X_val).clip(min=0)
        mse = mean_squared_error(lc_val, pred)
        if mse < best_glm_lc_mse:
            best_glm_lc_mse = mse
            best_glm_lc = glm
            best_glm_lc_params = {'power': p, 'alpha': alpha}

results['GLM_LC'] = best_glm_lc_mse
print(f'Best GLM LC  | MSE: {best_glm_lc_mse:.4f} | Params: {best_glm_lc_params}')

### 3b. XGBoost with Tweedie Loss

In [ ]:
param_dist_xgb = {
    'tweedie_variance_power': [1.2, 1.5, 1.8],
    'max_depth':              [3, 5, 7],
    'learning_rate':          [0.05, 0.1, 0.2],
    'n_estimators':           [200, 400],
    'subsample':              [0.7, 0.8, 1.0],
    'colsample_bytree':       [0.7, 0.8, 1.0],
    'alpha':                  [0, 1, 5],
}

xgb_base_lc = xgb.XGBRegressor(objective='reg:tweedie', random_state=42, n_jobs=-1)

rs_xgb_lc = RandomizedSearchCV(
    xgb_base_lc, param_dist_xgb,
    n_iter=30,
    scoring='neg_mean_squared_error',
    cv=3, n_jobs=-1, verbose=1, random_state=42
)
rs_xgb_lc.fit(X_tr, lc_tr)

best_xgb_lc = rs_xgb_lc.best_estimator_
pred_xgb_lc = best_xgb_lc.predict(X_val).clip(min=0)
mse_xgb_lc  = mean_squared_error(lc_val, pred_xgb_lc)

results['XGB_LC'] = mse_xgb_lc
print(f'Best XGB LC  | MSE: {mse_xgb_lc:.4f} | Params: {rs_xgb_lc.best_params_}')

### 3c. LightGBM with Tweedie Objective

In [ ]:
param_dist_lgbm = {
    'num_leaves':             [15, 31, 63],
    'min_data_in_leaf':       [20, 50, 100],
    'max_depth':              [3, 5, 7],
    'learning_rate':          [0.05, 0.1],
    'tweedie_variance_power': [1.2, 1.5, 1.8],
}

lgbm_base_lc = LGBMRegressor(
    objective='tweedie',
    n_estimators=500,
    random_state=42,
    verbose=-1
)

rs_lgbm_lc = RandomizedSearchCV(
    lgbm_base_lc, param_dist_lgbm,
    n_iter=30,
    scoring='neg_mean_squared_error',
    cv=3, n_jobs=-1, verbose=1, random_state=42
)
rs_lgbm_lc.fit(X_tr, lc_tr)

best_lgbm_lc = rs_lgbm_lc.best_estimator_
pred_lgbm_lc = best_lgbm_lc.predict(X_val).clip(min=0)
mse_lgbm_lc  = mean_squared_error(lc_val, pred_lgbm_lc)

results['LGBM_LC'] = mse_lgbm_lc
print(f'Best LGBM LC | MSE: {mse_lgbm_lc:.4f} | Params: {rs_lgbm_lc.best_params_}')

### 3d. Ensemble: Average of XGBoost + LightGBM

In [ ]:
pred_ens_lc = (pred_xgb_lc + pred_lgbm_lc) / 2
mse_ens_lc = mean_squared_error(lc_val, pred_ens_lc)
results['Ensemble_LC'] = mse_ens_lc
print(f'Ensemble LC  | MSE: {mse_ens_lc:.4f}')

### 3e. LC Model Comparison

In [ ]:
lc_compare = {k: v for k, v in results.items() if 'LC' in k}
lc_df = pd.DataFrame(lc_compare.items(), columns=['Model', 'Val MSE']).sort_values('Val MSE')
print('\n=== LC Model Comparison (lower is better) ===')
print(lc_df.to_string(index=False))

best_lc_model_name = lc_df.iloc[0]['Model']
print(f'\nSelected model for LC: {best_lc_model_name}')

## 4. Model Training: HALC

HALC = LC × X.18 (historical claim ratio). We model it directly as a separate Tweedie regression rather than post-multiplying LC predictions, since X.18 is not available in the test set.

### 4a. Tweedie GLM — Baseline

In [ ]:
best_glm_halc_mse = np.inf
best_glm_halc = None
best_glm_halc_params = {}

for p in [1.1, 1.3, 1.5, 1.7, 1.9]:
    for alpha in [0.001, 0.01, 0.1, 1.0]:
        glm = TweedieRegressor(power=p, alpha=alpha, max_iter=2000, link='log')
        glm.fit(X_tr, halc_tr)
        pred = glm.predict(X_val).clip(min=0)
        mse = mean_squared_error(halc_val, pred)
        if mse < best_glm_halc_mse:
            best_glm_halc_mse = mse
            best_glm_halc = glm
            best_glm_halc_params = {'power': p, 'alpha': alpha}

results['GLM_HALC'] = best_glm_halc_mse
print(f'Best GLM HALC  | MSE: {best_glm_halc_mse:.4f} | Params: {best_glm_halc_params}')

### 4b. XGBoost with Tweedie Loss

In [ ]:
xgb_base_halc = xgb.XGBRegressor(objective='reg:tweedie', random_state=42, n_jobs=-1)

rs_xgb_halc = RandomizedSearchCV(
    xgb_base_halc, param_dist_xgb,
    n_iter=30,
    scoring='neg_mean_squared_error',
    cv=3, n_jobs=-1, verbose=1, random_state=42
)
rs_xgb_halc.fit(X_tr, halc_tr)

best_xgb_halc = rs_xgb_halc.best_estimator_
pred_xgb_halc = best_xgb_halc.predict(X_val).clip(min=0)
mse_xgb_halc  = mean_squared_error(halc_val, pred_xgb_halc)

results['XGB_HALC'] = mse_xgb_halc
print(f'Best XGB HALC  | MSE: {mse_xgb_halc:.4f} | Params: {rs_xgb_halc.best_params_}')

### 4c. LightGBM with Tweedie Objective

In [ ]:
lgbm_base_halc = LGBMRegressor(
    objective='tweedie',
    n_estimators=500,
    random_state=42,
    verbose=-1
)

rs_lgbm_halc = RandomizedSearchCV(
    lgbm_base_halc, param_dist_lgbm,
    n_iter=30,
    scoring='neg_mean_squared_error',
    cv=3, n_jobs=-1, verbose=1, random_state=42
)
rs_lgbm_halc.fit(X_tr, halc_tr)

best_lgbm_halc = rs_lgbm_halc.best_estimator_
pred_lgbm_halc = best_lgbm_halc.predict(X_val).clip(min=0)
mse_lgbm_halc  = mean_squared_error(halc_val, pred_lgbm_halc)

results['LGBM_HALC'] = mse_lgbm_halc
print(f'Best LGBM HALC | MSE: {mse_lgbm_halc:.4f} | Params: {rs_lgbm_halc.best_params_}')

### 4d. Ensemble: Average of XGBoost + LightGBM

In [ ]:
pred_ens_halc = (pred_xgb_halc + pred_lgbm_halc) / 2
mse_ens_halc = mean_squared_error(halc_val, pred_ens_halc)
results['Ensemble_HALC'] = mse_ens_halc
print(f'Ensemble HALC  | MSE: {mse_ens_halc:.4f}')

### 4e. HALC Model Comparison

In [ ]:
halc_compare = {k: v for k, v in results.items() if 'HALC' in k}
halc_df = pd.DataFrame(halc_compare.items(), columns=['Model', 'Val MSE']).sort_values('Val MSE')
print('\n=== HALC Model Comparison (lower is better) ===')
print(halc_df.to_string(index=False))

best_halc_model_name = halc_df.iloc[0]['Model']
print(f'\nSelected model for HALC: {best_halc_model_name}')

## 5. Full Comparison Summary

In [ ]:
all_df = pd.DataFrame(results.items(), columns=['Model', 'Val MSE']).sort_values('Val MSE')
print('=== All Models — Validation MSE ===')
print(all_df.to_string(index=False))

## 6. Retrain Best Models on Full Training Data

After selecting the best models, retrain on all available training data (no validation holdout) to maximize predictive power before generating test predictions.

In [ ]:
# ---- LC: retrain on full data with best params found above ----

final_xgb_lc = xgb.XGBRegressor(
    objective='reg:tweedie',
    **{k: v for k, v in rs_xgb_lc.best_params_.items()},
    random_state=42, n_jobs=-1
)
final_xgb_lc.fit(X_full, LC)

final_lgbm_lc = LGBMRegressor(
    objective='tweedie',
    n_estimators=500,
    random_state=42,
    verbose=-1,
    **{k: v for k, v in rs_lgbm_lc.best_params_.items()}
)
final_lgbm_lc.fit(X_full, LC)

final_glm_lc = TweedieRegressor(
    power=best_glm_lc_params['power'],
    alpha=best_glm_lc_params['alpha'],
    max_iter=2000, link='log'
)
final_glm_lc.fit(X_full, LC)

print('LC models retrained on full data.')

In [ ]:
# ---- HALC: retrain on full data with best params found above ----

final_xgb_halc = xgb.XGBRegressor(
    objective='reg:tweedie',
    **{k: v for k, v in rs_xgb_halc.best_params_.items()},
    random_state=42, n_jobs=-1
)
final_xgb_halc.fit(X_full, HALC)

final_lgbm_halc = LGBMRegressor(
    objective='tweedie',
    n_estimators=500,
    random_state=42,
    verbose=-1,
    **{k: v for k, v in rs_lgbm_halc.best_params_.items()}
)
final_lgbm_halc.fit(X_full, HALC)

final_glm_halc = TweedieRegressor(
    power=best_glm_halc_params['power'],
    alpha=best_glm_halc_params['alpha'],
    max_iter=2000, link='log'
)
final_glm_halc.fit(X_full, HALC)

print('HALC models retrained on full data.')

In [ ]:
import joblib
import os

# Define the directory path
output_dir = 'models/'

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

joblib.dump(final_lgbm_lc,   os.path.join(output_dir, 'final_lgbm_lc.pkl'))
joblib.dump(final_lgbm_halc, os.path.join(output_dir, 'final_lgbm_halc.pkl'))
print('Models saved!')

## 7. Generate Test Predictions

In [ ]:
# --- Select best LC model ---
# Change this to 'GLM', 'XGB', or 'Ensemble' based on comparison results above
LC_MODEL = 'Ensemble'   # <-- adjust after running section 3e
HALC_MODEL = 'Ensemble' # <-- adjust after running section 4e

def predict_lc(X, model_name):
    if model_name == 'GLM':
        return final_glm_lc.predict(X).clip(min=0)
    elif model_name == 'XGB':
        return final_xgb_lc.predict(X).clip(min=0)
    elif model_name == 'LGBM':
        return final_lgbm_lc.predict(X).clip(min=0)
    elif model_name == 'Ensemble':
        return ((final_xgb_lc.predict(X) + final_lgbm_lc.predict(X)) / 2).clip(min=0)

def predict_halc(X, model_name):
    if model_name == 'GLM':
        return final_glm_halc.predict(X).clip(min=0)
    elif model_name == 'XGB':
        return final_xgb_halc.predict(X).clip(min=0)
    elif model_name == 'LGBM':
        return final_lgbm_halc.predict(X).clip(min=0)
    elif model_name == 'Ensemble':
        return ((final_xgb_halc.predict(X) + final_lgbm_halc.predict(X)) / 2).clip(min=0)

lc_test_pred   = predict_lc(X_test, LC_MODEL)
halc_test_pred = predict_halc(X_test, HALC_MODEL)

print(f'LC   predictions — mean: {lc_test_pred.mean():.4f}  max: {lc_test_pred.max():.4f}')
print(f'HALC predictions — mean: {halc_test_pred.mean():.4f}  max: {halc_test_pred.max():.4f}')

In [ ]:
# Sanity check: distributions should be similar to training targets
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, pred, train_vals, name, color in zip(
    axes,
    [lc_test_pred, halc_test_pred],
    [LC.values, HALC.values],
    ['LC', 'HALC'],
    ['steelblue', 'coral']
):
    nz_pred  = pred[pred > 0]
    nz_train = train_vals[train_vals > 0]
    ax.hist(nz_train, bins=50, alpha=0.5, color='gray', label='Train')
    ax.hist(nz_pred,  bins=50, alpha=0.7, color=color,  label='Test Pred')
    ax.set_title(f'{name} — Train vs Test Predictions (non-zero)')
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Build Final Submission CSV

Combine LC and HALC predictions with the CS predictions from the classification model.

In [ ]:
# Load CS predictions from classification notebook
cs_df = pd.read_csv('outputs/CS_predictions.csv')
print('CS predictions shape:', cs_df.shape)
print(cs_df.head())

# CS column: the classifier saved probabilities — threshold at 0.5 for binary 0/1
# Note: if the grader evaluates with ROC-AUC, keep probabilities; for binary labels, threshold below.
# For this submission, we use binary 0/1 as the project description specifies CS ∈ {0, 1}.
cs_binary = (cs_df['CS'].values >= 0.5).astype(int)

print(f'\nCS claim rate in predictions: {cs_binary.mean():.2%}')

In [ ]:
# Verify all lengths match
assert len(lc_test_pred) == len(halc_test_pred) == len(cs_binary), \
    f'Length mismatch: LC={len(lc_test_pred)}, HALC={len(halc_test_pred)}, CS={len(cs_binary)}'

submission = pd.DataFrame({
    'LC':   lc_test_pred,
    'HALC': halc_test_pred,
    'CS':   cs_binary
})

print(submission.shape)
print(submission.describe().round(4))
print(submission.head(10))

In [ ]:
# Save locally and to Drive
# IMPORTANT: rename this file to your actual group name before submitting
OUT_PATH = 'outputs/final_predictions.csv'

submission.to_csv(OUT_PATH, index=False)
submission.to_csv('GroupName_prediction.csv', index=False)

print(f'Saved {len(submission)} rows to {OUT_PATH}')